# Notebook 01 — Tox21 Exploratory Data Analysis

**Goal:** Understand the Tox21 dataset before modelling.  
Key questions:
1. What does the class imbalance look like per assay?
2. What is the molecular weight / LogP distribution of the compounds?
3. Are any tasks correlated (suggesting shared mechanism)?
4. Which structural alerts are already present in the training set?

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import deepchem as dc
from rdkit import Chem
from rdkit.Chem import Descriptors, Draw
from IPython.display import display

import sys
sys.path.insert(0, '..')
from src.data.tox21_loader import Tox21Loader, Tox21DataConfig, TOX21_TASKS, TASK_DESCRIPTIONS
from src.features.molecular_representations import MolecularRepresentation
from src.validation.chemistry_validator import ChemistryValidator

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
print('Libraries loaded ✓')

## 1. Load Dataset

In [ ]:
loader = Tox21Loader(Tox21DataConfig(featurizer_type='ECFP', splitter_type='scaffold'))
bundle = loader.load()

summary = loader.summary(bundle)
display(summary.style.background_gradient(subset=['pos_ratio'], cmap='RdYlGn'))
print(f"\nTrain: {len(bundle.train)} | Valid: {len(bundle.valid)} | Test: {len(bundle.test)}")

## 2. Class Imbalance — Per Task

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
colors = ['#d62728' if r < 0.05 else '#ff7f0e' if r < 0.10 else '#2ca02c'
          for r in summary['pos_ratio']]
bars = ax.bar(summary['task'], summary['pos_ratio'] * 100, color=colors, edgecolor='white', linewidth=0.8)
ax.axhline(5,  color='red',    linestyle='--', alpha=0.6, label='5% threshold (severe)')
ax.axhline(10, color='orange', linestyle='--', alpha=0.6, label='10% threshold (moderate)')
ax.set_ylabel('Positive Rate (%)')
ax.set_title('Tox21 Class Imbalance per Assay (scaffold split, train set)')
ax.legend()
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.savefig('../results/figures/class_imbalance.png', bbox_inches='tight')
plt.show()
print('Interpretation: Tasks below 5% require AUC-ROC / balanced accuracy as primary metric — accuracy is meaningless.')

## 3. Physicochemical Property Distributions

Understanding the chemical space of Tox21 helps interpret model failure modes.  
Heavy outliers in MW or LogP often correspond to aggregators or PAINS compounds.

In [ ]:
repr_calc = MolecularRepresentation()
all_smiles = bundle.train.ids.tolist()[:2000]  # subsample for speed

phys_df = repr_calc.batch_physicochemical(all_smiles).dropna()
print(f"Computed descriptors for {len(phys_df)} molecules")
display(phys_df.describe().round(2))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
props = [('MW', 'Molecular Weight (Da)'), ('LogP', 'Wildman-Crippen LogP'),
         ('TPSA', 'Topological PSA (Å²)'), ('HBD', 'H-bond donors'),
         ('HBA', 'H-bond acceptors'), ('Fsp3', 'Fraction Csp3')]

for ax, (col, label) in zip(axes.flat, props):
    data = phys_df[col].clip(phys_df[col].quantile(0.01), phys_df[col].quantile(0.99))
    ax.hist(data, bins=50, color='#4C72B0', alpha=0.85, edgecolor='white', linewidth=0.3)
    ax.set_xlabel(label)
    ax.set_ylabel('Count')
    ax.axvline(data.median(), color='#DD4444', linestyle='--', linewidth=1.5,
               label=f'Median: {data.median():.1f}')
    ax.legend(fontsize=9)

fig.suptitle('Tox21 Chemical Space — Physicochemical Properties (n=2000)', fontsize=13)
plt.tight_layout()
plt.savefig('../results/figures/physicochemical_distributions.png', bbox_inches='tight')
plt.show()

## 4. Inter-Task Label Correlation

Correlated tasks suggest shared mechanisms — the GNN can exploit this via multi-task learning.  
The two NR-ER tasks should be highly correlated (same protein, different assay formats).

In [ ]:
y_train = bundle.train.y
w_train = bundle.train.w

# Build a masked label matrix (NaN where w=0)
y_masked = np.where(w_train == 0, np.nan, y_train)
label_df = pd.DataFrame(y_masked, columns=TOX21_TASKS)

# Correlation on labelled pairs only
corr = label_df.corr(method='spearman', min_periods=100)

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-0.5, vmax=0.5, ax=ax,
            linewidths=0.3, annot_kws={'size': 9})
ax.set_title('Spearman Label Correlation Between Tox21 Assays\n(multi-task learning can exploit positive off-diagonals)')
plt.tight_layout()
plt.savefig('../results/figures/task_correlation.png', bbox_inches='tight')
plt.show()
print('NR-ER / NR-ER-LBD high correlation expected — both target estrogen receptor alpha.')

## 5. Pre-model Structural Alert Prevalence

Before training anything, how many training-set compounds carry known toxicophores?  
This sets a chemical floor for expected positive rates in the validation layer.

In [ ]:
validator = ChemistryValidator()
sample_smiles = bundle.train.ids.tolist()[:500]  # subsample

alert_counts = {}
for smi in sample_smiles:
    for alert in validator.screen_molecule(smi):
        alert_counts[alert.name] = alert_counts.get(alert.name, 0) + 1

alert_series = pd.Series(alert_counts).sort_values(ascending=True)
alert_pct    = (alert_series / len(sample_smiles) * 100).round(1)

fig, ax = plt.subplots(figsize=(10, 5))
alert_pct.tail(12).plot(kind='barh', ax=ax, color='#e05c47', edgecolor='white')
ax.set_xlabel('% of training compounds containing alert')
ax.set_title(f'Structural Alert Prevalence in Tox21 Training Set (n={len(sample_smiles)})')
for patch, val in zip(ax.patches, alert_pct.tail(12)):
    ax.text(patch.get_width() + 0.2, patch.get_y() + patch.get_height()/2,
            f'{val}%', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('../results/figures/alert_prevalence.png', bbox_inches='tight')
plt.show()
print(f'\nTotal alerts detected across {len(sample_smiles)} compounds: {sum(alert_counts.values())}')

## 6. Lipinski / Druglikeness Flags

Compounds violating Lipinski's Rule of 5 may have poor oral bioavailability —
relevant context for interpreting ADMET predictions.

In [ ]:
ro5_violations = (
    (phys_df['MW']  > 500).astype(int) +
    (phys_df['LogP'] > 5).astype(int)  +
    (phys_df['HBD'] > 5).astype(int)  +
    (phys_df['HBA'] > 10).astype(int)
)
ro5_df = ro5_violations.value_counts().sort_index()

fig, ax = plt.subplots(figsize=(7, 4))
ro5_df.plot(kind='bar', ax=ax, color=['#2ca02c','#ffbf00','#ff7f0e','#d62728','#8B0000'],
            edgecolor='white')
ax.set_xlabel('Number of Ro5 Violations')
ax.set_ylabel('Compound Count')
ax.set_title("Lipinski Rule-of-5 Compliance (Tox21 training set)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
pct_compliant = (ro5_violations <= 1).mean() * 100
print(f'{pct_compliant:.1f}% of compounds have ≤1 Ro5 violation (druglike)')